In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
from pygam import LinearGAM, s, te

# transformer package (pytorch)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
# importing and browsing data
csv_path = Path("..") / "Dataset" / "Outputs" / "CS2_model_input.csv"
df = pd.read_csv(csv_path)

print("Path:", csv_path.resolve())
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nDtypes:\n", df.dtypes)
display(df.head(5))

In [ ]:
# check wildfire related data
wildfire_cols = [
    "fire_count_regional",
    "frp_regional_sum",
    "hfi_weighted",
    "fwi_mean",
    "fire_count_local",
    "frp_local_sum",
]
for c in wildfire_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# counts + percents of non-zero (treat NaN as zero for the non-zero test)
nonzero_counts = df.groupby("Zone")[wildfire_cols].apply(lambda g: (g.fillna(0) != 0).sum())
row_counts = df.groupby("Zone").size().rename("n_rows")

nonzero_pct = (nonzero_counts.div(row_counts, axis=0) * 100).round(2)
nonzero_pct.columns = [f"{c}_pct" for c in nonzero_pct.columns]

summary = pd.concat([row_counts, nonzero_counts, nonzero_pct], axis=1).sort_index()

display(summary)

In [ ]:
# Baseline Modeling Preparation
WILDFIRE_COLS = [
    "fire_count_regional",
    "frp_regional_sum",
    "hfi_weighted",
    "fwi_mean",
    "fire_count_local",
    "frp_local_sum",
]

BASE_COLS = [
    "PM25_lag_24h",
    "PM25_lag_48h",
    "lag3",          # we will create this
    "hour",
    "day_of_week",
    "month",
]

ENHANCED_COLS = BASE_COLS + WILDFIRE_COLS

def make_zone_table(df_zone: pd.DataFrame) -> pd.DataFrame:
    d = df_zone.copy()
    # ✅ force datetime
    d["Datetime_UTC"] = pd.to_datetime(d["Datetime_UTC"], utc=True, errors="coerce")

    d = d.sort_values("Datetime_UTC")
    d["PM25"] = pd.to_numeric(d["PM25"], errors="coerce")

    # create lag3 from PM25 (3 hours back within zone)
    d["lag3"] = d["PM25"].shift(3)

    # ensure numeric wildfire cols
    for c in WILDFIRE_COLS:
        d[c] = pd.to_numeric(d[c], errors="coerce")

    # calendar cols should already exist, but ensure numeric
    for c in ["hour", "day_of_week", "month"]:
        d[c] = pd.to_numeric(d[c], errors="coerce")

    # drop rows with missing in any needed columns later (done after selecting features)
    return d

def fit_gam_and_eval(
    d: pd.DataFrame,
    feature_cols: list[str],
    train_years=(2022, 2023, 2024),
    test_year=2025,
    threshold=25.0,
):
    # select rows for train/test
    d = d.copy()
    d["year"] = d["Datetime_UTC"].dt.year

    # drop NA for selected features + target
    d_model = d.dropna(subset=feature_cols + ["PM25"]).copy()

    train = d_model[d_model["year"].isin(train_years)]
    test = d_model[d_model["year"].eq(test_year)]

    if len(train) < 500 or len(test) < 100:
        raise ValueError(f"Too few rows: train={len(train)}, test={len(test)}")

    X_train = train[feature_cols].values
    y_train = train["PM25"].values

    X_test = test[feature_cols].values
    y_test = test["PM25"].values
    idx_test = test["Datetime_UTC"].values

    # ---- GAM specification ----
    # Keep the CS1 “feel”: smooth lags + hour interactions.
    # For variable number of features, we build terms dynamically.
    # - smooth for each numeric predictor
    # - plus te(hour, day_of_week) and te(hour, month) if those columns exist
    col_idx = {c: i for i, c in enumerate(feature_cols)}

    terms = None
    # smooth for all features by default
    for i in range(len(feature_cols)):
        terms = s(i) if terms is None else terms + s(i)

    # add CS1-style tensor interactions if the columns exist
    if "hour" in col_idx and "day_of_week" in col_idx:
        terms = terms + te(col_idx["hour"], col_idx["day_of_week"], n_splines=[12, 7])
    if "hour" in col_idx and "month" in col_idx:
        terms = terms + te(col_idx["hour"], col_idx["month"], n_splines=[12, 6])

    gam = LinearGAM(terms).fit(X_train, y_train)

    y_pred = gam.predict(X_test)

    # regression metrics
    rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    mae = float(mean_absolute_error(y_test, y_pred))
    r2 = float(r2_score(y_test, y_pred))

    # event metrics: exceedance of threshold
    y_true_bin = (y_test > threshold).astype(int)
    y_pred_bin = (y_pred > threshold).astype(int)

    prec = float(precision_score(y_true_bin, y_pred_bin, zero_division=0))
    rec = float(recall_score(y_true_bin, y_pred_bin, zero_division=0))
    f1 = float(f1_score(y_true_bin, y_pred_bin, zero_division=0))

    metrics = {
        "N_train": int(len(train)),
        "N_test": int(len(test)),
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        f"Precision@{threshold:g}": prec,
        f"Recall@{threshold:g}": rec,
        f"F1@{threshold:g}": f1,
        "ActualEvents": int(y_true_bin.sum()),
        "PredEvents": int(y_pred_bin.sum()),
        "GAM_PseudoR2_ExplDev": float(gam.statistics_["pseudo_r2"]["explained_deviance"]),
    }

    pred_df = pd.DataFrame(
        {"Datetime_UTC": idx_test, "y_true": y_test, "y_pred": y_pred}
    ).sort_values("Datetime_UTC")

    return gam, metrics, pred_df

## baseline 1
Same logic as Case Study 1 but more simple. For each air zone, use 2022 - 2024 as training and 2025 as prediction

Baseline 1 use same input as CS 1 model

## baseline 2
Enhanced baseline model. For each air zone, use 2022 - 2024 as training and 2025 as prediction

Baseline 2 use additional features of wildfires than baseline 1

In [ ]:
# GAM training skipped — but still define zones for later cells
zones = sorted(df["Zone"].unique())

In [ ]:
def plot_test_series(pred_base: pd.DataFrame, pred_enh: pd.DataFrame, zone: str, max_hours=24*21):
    # merge on datetime (inner join so both have same timestamps)
    a = pred_base.rename(columns={"y_pred": "y_pred_baseline"})[["Datetime_UTC", "y_true", "y_pred_baseline"]]
    b = pred_enh.rename(columns={"y_pred": "y_pred_enhanced"})[["Datetime_UTC", "y_pred_enhanced"]]
    m = a.merge(b, on="Datetime_UTC", how="inner").sort_values("Datetime_UTC")

    # optionally plot only first window for readability
    if max_hours is not None and len(m) > max_hours:
        m = m.iloc[:max_hours]

    plt.figure(figsize=(14, 5))
    plt.plot(m["Datetime_UTC"], m["y_true"], label="Actual", linewidth=1.2)
    plt.plot(m["Datetime_UTC"], m["y_pred_baseline"], label="Baseline (CS1 features)", linewidth=1.1)
    plt.plot(m["Datetime_UTC"], m["y_pred_enhanced"], label="Enhanced (+ wildfire)", linewidth=1.1)
    plt.title(f"{zone} | 2025 Test: Actual vs Predictions (first {len(m)} hours)")
    plt.xlabel("Datetime (UTC)")
    plt.ylabel("PM2.5")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

for z in zones:
    plot_test_series(preds_by_zone[z]["baseline"], preds_by_zone[z]["enhanced"], zone=z, max_hours=24*30)

In [ ]:
# check wildfire season
THRESHOLD = 25.0

def filter_test_wildfire_season(d_model: pd.DataFrame, test_year=2025):
    d2 = d_model.copy()
    d2["year"] = d2["Datetime_UTC"].dt.year
    test = d2[d2["year"].eq(test_year)].copy()
    test_ws = test[test["is_wildfire_season"].eq(1)].copy()
    return test_ws

def eval_on_subset(gam, test_df: pd.DataFrame, feature_cols: list[str], threshold=25.0):
    X = test_df[feature_cols].values
    y = test_df["PM25"].values
    yhat = gam.predict(X)

    rmse = float(np.sqrt(mean_squared_error(y, yhat)))
    mae  = float(mean_absolute_error(y, yhat))
    r2   = float(r2_score(y, yhat))

    y_true_bin = (y > threshold).astype(int)
    y_pred_bin = (yhat > threshold).astype(int)

    prec = float(precision_score(y_true_bin, y_pred_bin, zero_division=0))
    rec  = float(recall_score(y_true_bin, y_pred_bin, zero_division=0))
    f1   = float(f1_score(y_true_bin, y_pred_bin, zero_division=0))

    return {
        "N_test_wildfire_season": int(len(test_df)),
        "RMSE_WS": rmse,
        "MAE_WS": mae,
        "R2_WS": r2,
        f"Precision@{threshold:g}_WS": prec,
        f"Recall@{threshold:g}_WS": rec,
        f"F1@{threshold:g}_WS": f1,
        "ActualEvents_WS": int(y_true_bin.sum()),
        "PredEvents_WS": int(y_pred_bin.sum()),
    }, pd.DataFrame({"Datetime_UTC": test_df["Datetime_UTC"].values,
                    "y_true": y,
                    "y_pred": yhat}).sort_values("Datetime_UTC")
zones = sorted(df["Zone"].unique())

all_rows_ws = []
preds_ws_by_zone = {}
models_by_zone = {}

for z in zones:
    dz = make_zone_table(df[df["Zone"] == z])

    # Fit on full 2022-2024 train (same as before)
    gam_base, m_base, p_base = fit_gam_and_eval(dz, BASE_COLS, threshold=THRESHOLD)
    gam_enh,  m_enh,  p_enh  = fit_gam_and_eval(dz, ENHANCED_COLS, threshold=THRESHOLD)

    models_by_zone[z] = {"baseline": gam_base, "enhanced": gam_enh}

    # Build the same modeling table used in training (dropna happens inside eval)
    d_model_base = dz.dropna(subset=BASE_COLS + ["PM25"]).copy()
    d_model_enh  = dz.dropna(subset=ENHANCED_COLS + ["PM25"]).copy()

    # Filter to 2025 wildfire season
    test_ws_base = filter_test_wildfire_season(d_model_base, test_year=2025)
    test_ws_enh  = filter_test_wildfire_season(d_model_enh,  test_year=2025)

    # Evaluate both models on wildfire season subset
    met_ws_base, pred_ws_base = eval_on_subset(gam_base, test_ws_base, BASE_COLS, threshold=THRESHOLD)
    met_ws_enh,  pred_ws_enh  = eval_on_subset(gam_enh,  test_ws_enh,  ENHANCED_COLS, threshold=THRESHOLD)

    all_rows_ws.append({"Zone": z, "Model": "Baseline_CS1Features_GAM", **met_ws_base})
    all_rows_ws.append({"Zone": z, "Model": "Enhanced_Wildfire_GAM", **met_ws_enh})

    preds_ws_by_zone[z] = {"baseline": pred_ws_base, "enhanced": pred_ws_enh}

metrics_ws_df = pd.DataFrame(all_rows_ws).sort_values(["Zone", "Model"]).reset_index(drop=True)
display(metrics_ws_df)


In [ ]:
def plot_wildfire_season_series(pred_base: pd.DataFrame, pred_enh: pd.DataFrame, zone: str, threshold=25.0):
    a = pred_base.rename(columns={"y_pred": "y_pred_baseline"})[["Datetime_UTC", "y_true", "y_pred_baseline"]]
    b = pred_enh.rename(columns={"y_pred": "y_pred_enhanced"})[["Datetime_UTC", "y_pred_enhanced"]]
    m = a.merge(b, on="Datetime_UTC", how="inner").sort_values("Datetime_UTC")

    plt.figure(figsize=(14, 5))
    plt.plot(m["Datetime_UTC"], m["y_true"], label="Actual", linewidth=1.2)
    plt.plot(m["Datetime_UTC"], m["y_pred_baseline"], label="Baseline (CS1 features)", linewidth=1.1)
    plt.plot(m["Datetime_UTC"], m["y_pred_enhanced"], label="Enhanced (+ wildfire)", linewidth=1.1)
    plt.axhline(threshold, linestyle="--", linewidth=1, label=f"Threshold={threshold}")
    plt.title(f"{zone} | 2025 Wildfire Season Only: Actual vs Predictions")
    plt.xlabel("Datetime (UTC)")
    plt.ylabel("PM2.5")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

for z in zones:
    plot_wildfire_season_series(
        preds_ws_by_zone[z]["baseline"],
        preds_ws_by_zone[z]["enhanced"],
        zone=z,
        threshold=THRESHOLD
    )

## A simple try for transformer (pytorch)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

csv_path = Path("..") / "Dataset" / "Outputs" / "CS2_model_input.csv"  # notebook in /Model
df = pd.read_csv(csv_path)

df["Datetime_UTC"] = pd.to_datetime(df["Datetime_UTC"], utc=True, errors="coerce")
df = df.dropna(subset=["Datetime_UTC"]).sort_values(["Zone", "Datetime_UTC"]).reset_index(drop=True)

# Ensure numeric where needed
df["PM25"] = pd.to_numeric(df["PM25"], errors="coerce")

WILDFIRE_COLS = [
    "fire_count_regional", "frp_regional_sum", "hfi_weighted", "fwi_mean",
    "fire_count_local", "frp_local_sum",
]

BASE_COLS = [
    "PM25_lag_1h", "PM25_lag_6h", "PM25_lag_24h", "PM25_lag_48h",
    "PM25_roll_24h_mean", "PM25_roll_24h_max", "PM25_roll_7d_mean",
    "frp_roll_24h_sum", "frp_roll_72h_sum",
    "hour", "day_of_week", "month", "is_wildfire_season",
]

FEATURES = BASE_COLS + WILDFIRE_COLS
TARGET = "PM25"

In [ ]:
SEQ_LEN = 48      # past 48 hours
HORIZON = 3       # predict 3 hours ahead

def build_zone_frame(df_zone: pd.DataFrame) -> pd.DataFrame:
    d = df_zone.copy().sort_values("Datetime_UTC")
    # coerce features to numeric
    for c in FEATURES:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d[TARGET] = pd.to_numeric(d[TARGET], errors="coerce")
    d = d.dropna(subset=FEATURES + [TARGET])
    d["year"] = d["Datetime_UTC"].dt.year
    return d

def make_sequences(X: np.ndarray, y: np.ndarray, seq_len: int, horizon: int):
    # X shape: (T, F), y shape: (T,)
    T = len(X)
    Xs, ys = [], []
    # last usable index t is T - horizon - 1
    for t in range(seq_len - 1, T - horizon):
        Xs.append(X[t - seq_len + 1 : t + 1])
        ys.append(y[t + horizon])
    return np.stack(Xs), np.array(ys)

In [ ]:
class SeqDataset(Dataset):
    def __init__(self, X_seq, y):
        self.X = torch.tensor(X_seq, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.X[i], self.y[i]

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # x: (B, L, D)
        L = x.size(1)
        return x + self.pe[:, :L, :]

class TransformerRegressor(nn.Module):
    def __init__(self, n_features, d_model=64, n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos = PositionalEncoding(d_model, max_len=512)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4*d_model,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, 1),
        )

    def forward(self, x):
        # x: (B, L, F)
        h = self.input_proj(x)         # (B, L, D)
        h = self.pos(h)
        h = self.encoder(h)            # (B, L, D)
        h_last = h[:, -1, :]           # last timestep representation
        out = self.head(h_last).squeeze(-1)  # (B,)
        return out

In [ ]:
def train_one_zone(dz: pd.DataFrame, zone_name: str, epochs=10, batch_size=256, lr=1e-3):
    # split
    train_df = dz[dz["year"].isin([2022, 2023, 2024])].copy()
    test_df  = dz[dz["year"].eq(2025)].copy()

    # scale using TRAIN only
    scaler = StandardScaler()
    X_train_raw = scaler.fit_transform(train_df[FEATURES].values)
    y_train_raw = train_df[TARGET].values

    X_test_raw = scaler.transform(test_df[FEATURES].values)
    y_test_raw = test_df[TARGET].values

    # sequences
    Xtr_seq, ytr = make_sequences(X_train_raw, y_train_raw, SEQ_LEN, HORIZON)
    Xte_seq, yte = make_sequences(X_test_raw,  y_test_raw,  SEQ_LEN, HORIZON)

    train_loader = DataLoader(SeqDataset(Xtr_seq, ytr), batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=(device.type=="cuda"))
    test_loader  = DataLoader(SeqDataset(Xte_seq, yte), batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=(device.type=="cuda"))

    model = TransformerRegressor(n_features=len(FEATURES), d_model=64, n_heads=4, n_layers=2, dropout=0.1).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.MSELoss()

    use_amp = (device.type == "cuda")
    scaler_amp = torch.cuda.amp.GradScaler(enabled=use_amp)

    for ep in range(1, epochs+1):
        model.train()
        total = 0.0
        n = 0

        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                pred = model(xb)
                loss = loss_fn(pred, yb)

            scaler_amp.scale(loss).backward()
            scaler_amp.step(opt)
            scaler_amp.update()

            total += loss.item() * len(yb)
            n += len(yb)

        train_mse = total / max(n, 1)

        # quick eval
        model.eval()
        preds = []
        trues = []
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(device, non_blocking=True)
                pred = model(xb).detach().cpu().numpy()
                preds.append(pred)
                trues.append(yb.numpy())

        yhat = np.concatenate(preds)
        ytrue = np.concatenate(trues)

        rmse = float(np.sqrt(mean_squared_error(ytrue, yhat)))
        mae  = float(mean_absolute_error(ytrue, yhat))
        r2   = float(r2_score(ytrue, yhat))

        print(f"[{zone_name}] epoch {ep:02d} | train_mse={train_mse:.4f} | test RMSE={rmse:.4f} MAE={mae:.4f} R2={r2:.4f}")

    # return final predictions aligned to test timestamps (for plotting)
    # rebuild timestamps for test sequences:
    test_times = test_df["Datetime_UTC"].values
    # sequence targets start at index (SEQ_LEN-1 + HORIZON)
    start_idx = (SEQ_LEN - 1 + HORIZON)
    pred_times = test_times[start_idx : start_idx + len(yhat)]

    pred_df = pd.DataFrame({"Datetime_UTC": pred_times, "y_true": ytrue, "y_pred": yhat}).sort_values("Datetime_UTC")
    metrics = {"Zone": zone_name, "RMSE": rmse, "MAE": mae, "R2": r2, "N_test": int(len(pred_df))}

    return model, scaler, metrics, pred_df

In [ ]:
# v1 transformer training — skipped (not needed for augmentation comparison)
# zones = sorted(df["Zone"].unique())
# results = []
# ...
pass

In [ ]:
def plot_transformer(pred_df: pd.DataFrame, zone: str, max_hours=24*21):
    d = pred_df.sort_values("Datetime_UTC").copy()
    if max_hours is not None and len(d) > max_hours:
        d = d.iloc[:max_hours]

    plt.figure(figsize=(14, 5))
    plt.plot(d["Datetime_UTC"], d["y_true"], label="Actual", linewidth=1.2)
    plt.plot(d["Datetime_UTC"], d["y_pred"], label="Transformer", linewidth=1.1)
    plt.title(f"{zone} | 2025 Test: Actual vs Transformer (first {len(d)} points)")
    plt.xlabel("Datetime (UTC)")
    plt.ylabel("PM2.5")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

for z in zones:
    plot_transformer(preds_by_zone[z], z, max_hours=24*21)

## Raw-history Transformer(ignore added lag and do simple hyperparameter tuning)

In [ ]:
THRESHOLD = 25.0

RAW_FEATURES = [
    "PM25",
    "fire_count_regional", "frp_regional_sum", "hfi_weighted", "fwi_mean",
    "fire_count_local", "frp_local_sum",
    "hour", "day_of_week", "month", "is_wildfire_season",
]
TARGET = "PM25"
csv_path = Path("..") / "Dataset" / "Outputs" / "CS2_model_input.csv"  # notebook in /Model
df = pd.read_csv(csv_path)

df["Datetime_UTC"] = pd.to_datetime(df["Datetime_UTC"], utc=True, errors="coerce")
df = df.dropna(subset=["Datetime_UTC"]).sort_values(["Zone", "Datetime_UTC"]).reset_index(drop=True)

def build_zone_frame_raw(df_zone: pd.DataFrame) -> pd.DataFrame:
    d = df_zone.copy().sort_values("Datetime_UTC")
    for c in RAW_FEATURES + [TARGET]:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d = d.dropna(subset=RAW_FEATURES + [TARGET])
    d["year"] = d["Datetime_UTC"].dt.year
    return d

class SeqDataset(Dataset):
    def __init__(self, X_seq, y, t, wildfire_flag=None):
        self.X = torch.tensor(X_seq, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.t = t
        self.ws = wildfire_flag  # numpy array aligned to y (optional)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.X[i], self.y[i], i

def make_sequences_from_frame(d: pd.DataFrame, features, target, seq_len, horizon):
    X = d[features].values  # (T,F)
    y = d[target].values
    t = d["Datetime_UTC"].values
    ws = d["is_wildfire_season"].values.astype(int)

    Xs, ys, ts, wss = [], [], [], []
    for idx in range(seq_len - 1, len(d) - horizon):
        Xs.append(X[idx - seq_len + 1: idx + 1])
        ys.append(y[idx + horizon])
        ts.append(t[idx + horizon])
        wss.append(ws[idx + horizon])

    return np.stack(Xs), np.array(ys), np.array(ts), np.array(wss)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1024):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # [1, max_len, d_model]

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class TransformerRegressor(nn.Module):
    def __init__(self, n_features, d_model=64, nhead=4, num_layers=2, dim_ff=256, dropout=0.1):
        super().__init__()
        self.in_proj = nn.Linear(n_features, d_model)
        self.pos = PositionalEncoding(d_model)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True
        )
        self.enc = nn.TransformerEncoder(layer, num_layers=num_layers)

        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        # x: [B, T, F]
        h = self.in_proj(x)
        h = self.pos(h)
        h = self.enc(h)
        h_last = h[:, -1, :]
        return self.head(h_last).squeeze(-1)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1024):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # [1, max_len, d_model]

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class TransformerRegressor(nn.Module):
    def __init__(self, n_features, d_model=64, nhead=4, num_layers=2, dim_ff=256, dropout=0.1):
        super().__init__()
        self.in_proj = nn.Linear(n_features, d_model)
        self.pos = PositionalEncoding(d_model)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True
        )
        self.enc = nn.TransformerEncoder(layer, num_layers=num_layers)

        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        # x: [B, T, F]
        h = self.in_proj(x)
        h = self.pos(h)
        h = self.enc(h)
        h_last = h[:, -1, :]
        return self.head(h_last).squeeze(-1)

In [ ]:
def compute_metrics(y_true, y_pred, threshold=25.0):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))

    ytb = (y_true > threshold).astype(int)
    ypb = (y_pred > threshold).astype(int)
    prec = float(precision_score(ytb, ypb, zero_division=0))
    rec = float(recall_score(ytb, ypb, zero_division=0))
    f1v = float(f1_score(ytb, ypb, zero_division=0))
    return {"RMSE": rmse, "MAE": mae, "R2": r2,
            f"Precision@{threshold:g}": prec, f"Recall@{threshold:g}": rec, f"F1@{threshold:g}": f1v,
            "ActualEvents": int(ytb.sum()), "PredEvents": int(ypb.sum())}

def train_transformer_zone_raw(
    dz: pd.DataFrame,
    zone_name: str,
    seq_len=72,
    horizon=3,
    d_model=64,
    n_layers=2,
    nhead=4,
    batch_size=256,
    lr=1e-3,
    max_epochs=30,
    patience=5,
):
    # split
    train_df = dz[dz["year"].isin([2022, 2023, 2024])].copy()
    test_df  = dz[dz["year"].eq(2025)].copy()

    # ---- scale features using TRAIN only (do NOT write back to df) ----
    scaler = StandardScaler()

    X_train_raw = scaler.fit_transform(train_df[RAW_FEATURES].astype(float).values)
    y_train_raw = train_df[TARGET].astype(float).values
    ws_train = train_df["is_wildfire_season"].astype(int).values
    t_train = train_df["Datetime_UTC"].values

    X_test_raw = scaler.transform(test_df[RAW_FEATURES].astype(float).values)
    y_test_raw = test_df[TARGET].astype(float).values
    ws_test = test_df["is_wildfire_season"].astype(int).values
    t_test = test_df["Datetime_UTC"].values

    def make_sequences_from_arrays(X, y, t, ws, seq_len, horizon):
        Xs, ys, ts, wss = [], [], [], []
        for idx in range(seq_len - 1, len(y) - horizon):
            Xs.append(X[idx - seq_len + 1: idx + 1])
            ys.append(y[idx + horizon])
            ts.append(t[idx + horizon])
            wss.append(ws[idx + horizon])
        return np.stack(Xs), np.array(ys), np.array(ts), np.array(wss)

    Xtr, ytr, ttr, wtr = make_sequences_from_arrays(X_train_raw, y_train_raw, t_train, ws_train, seq_len, horizon)
    Xte, yte, tte, wte = make_sequences_from_arrays(X_test_raw,  y_test_raw,  t_test,  ws_test,  seq_len, horizon)

    # train/val split (last 10% as val to preserve time order)
    n = len(ytr)
    n_val = max(1, int(0.1 * n))
    split = n - n_val
    X_train, y_train = Xtr[:split], ytr[:split]
    X_val, y_val     = Xtr[split:], ytr[split:]

    train_loader = DataLoader(SeqDataset(X_train, y_train, None), batch_size=batch_size, shuffle=True,
                              pin_memory=(device.type=="cuda"))
    val_loader   = DataLoader(SeqDataset(X_val, y_val, None), batch_size=batch_size, shuffle=False,
                              pin_memory=(device.type=="cuda"))
    test_loader  = DataLoader(SeqDataset(Xte, yte, tte, wildfire_flag=wte), batch_size=batch_size, shuffle=False,
                              pin_memory=(device.type=="cuda"))

    model = TransformerRegressor(len(RAW_FEATURES), d_model=d_model, nhead=nhead, num_layers=n_layers, dim_ff=4*d_model).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.SmoothL1Loss()  # ✅ Huber
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=2)

    best_val = float("inf")
    best_state = None
    bad = 0

    use_amp = (device.type == "cuda")
    scaler_amp = torch.cuda.amp.GradScaler(enabled=use_amp)

    for ep in range(1, max_epochs + 1):
        model.train()
        tr_losses = []
        for xb, yb, _ in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=use_amp):
                pred = model(xb)
                loss = loss_fn(pred, yb)

            scaler_amp.scale(loss).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler_amp.step(opt)
            scaler_amp.update()
            tr_losses.append(loss.item())

        # val
        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb, _ in val_loader:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)
                pred = model(xb)
                val_losses.append(loss_fn(pred, yb).item())

        val_loss = float(np.mean(val_losses))
        sched.step(val_loss)

        print(f"[{zone_name}] ep {ep:02d} | train_loss {np.mean(tr_losses):.4f} | val_loss {val_loss:.4f} | lr {opt.param_groups[0]['lr']:.2e}")

        if val_loss < best_val - 1e-4:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print(f"[{zone_name}] Early stopping at epoch {ep}")
                break

    model.load_state_dict(best_state)

    # test preds
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, yb, _ in test_loader:
            xb = xb.to(device, non_blocking=True)
            preds.append(model(xb).detach().cpu().numpy())
    yhat = np.concatenate(preds)

    # overall metrics
    met_all = compute_metrics(yte, yhat, threshold=THRESHOLD)

    # wildfire season metrics
    mask_ws = (wte == 1)
    if mask_ws.sum() > 0:
        met_ws = compute_metrics(yte[mask_ws], yhat[mask_ws], threshold=THRESHOLD)
    else:
        met_ws = {"RMSE": np.nan, "MAE": np.nan, "R2": np.nan,
                  f"Precision@{THRESHOLD:g}": np.nan, f"Recall@{THRESHOLD:g}": np.nan, f"F1@{THRESHOLD:g}": np.nan,
                  "ActualEvents": 0, "PredEvents": 0}

    pred_df = pd.DataFrame({"Datetime_UTC": tte, "y_true": yte, "y_pred": yhat, "is_wildfire_season": wte}).sort_values("Datetime_UTC")

    return model, {"Zone": zone_name, "SEQ_LEN": seq_len, "HORIZON": horizon, **met_all}, {"Zone": zone_name, "SEQ_LEN": seq_len, "HORIZON": horizon, **{k+"_WS": v for k,v in met_ws.items()}}, pred_df

In [ ]:
zones = sorted(df["Zone"].unique())

# Best configs identified from prior grid search — run only these 2 instead of all 8
best_configs = {
    "Lower Fraser Valley": (72,  1),
    "Southern Interior":   (168, 1),
}

all_all = []
all_ws  = []
preds_best = {}

for z in zones:
    dz = build_zone_frame_raw(df[df["Zone"] == z])
    seq_len, horizon = best_configs[z]

    print(f"\nZONE={z} | SEQ_LEN={seq_len} | HORIZON={horizon}")
    model, met_all, met_ws, pred_df = train_transformer_zone_raw(
        dz, z, seq_len=seq_len, horizon=horizon,
        d_model=64, n_layers=2, nhead=4,
        batch_size=256, lr=1e-3, max_epochs=30, patience=5
    )

    all_all.append(met_all)
    all_ws.append(met_ws)
    preds_best[z] = {"seq_len": seq_len, "horizon": horizon, "pred_df": pred_df}

all_df = pd.DataFrame(all_all).sort_values(["Zone", "RMSE"]).reset_index(drop=True)
ws_df  = pd.DataFrame(all_ws).sort_values(["Zone", "RMSE_WS"]).reset_index(drop=True)

display(all_df)
display(ws_df)

In [ ]:
def plot_pred(pred_df, zone, horizon, seq_len, max_hours=24*21):
    d = pred_df.sort_values("Datetime_UTC").copy()
    if max_hours is not None and len(d) > max_hours:
        d = d.iloc[:max_hours]

    plt.figure(figsize=(14, 5))
    plt.plot(d["Datetime_UTC"], d["y_true"], label="Actual", linewidth=1.2)
    plt.plot(d["Datetime_UTC"], d["y_pred"], label="Transformer", linewidth=1.1)
    plt.axhline(THRESHOLD, linestyle="--", linewidth=1, label=f"Threshold={THRESHOLD}")
    plt.title(f"{zone} | 2025 Test | best Transformer (SEQ_LEN={seq_len}, H={horizon})")
    plt.xlabel("Datetime (UTC)")
    plt.ylabel("PM2.5")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

for z in zones:
    pack = preds_best[z]
    plot_pred(pack["pred_df"], z, horizon=pack["horizon"], seq_len=pack["seq_len"], max_hours=24*21)

## Augmented Training Comparison: PCHIP 30min / CVAE / TimeGAN

Training the same Transformer architecture on three augmented datasets.
Test is always **2025 real data** (PCHIP uses 30-min resolution; others use hourly).

In [ ]:

# ── Shared training utility for augmented experiments ─────────────────────────
# Reuses from earlier cells: SeqDataset, TransformerRegressor, compute_metrics,
#                            device, THRESHOLD, RAW_FEATURES, TARGET

from sklearn.preprocessing import StandardScaler as _SS

# Safety: ensure zones is defined even if earlier grid-search cell was skipped
if "zones" not in dir():
    zones = sorted(df["Zone"].unique())

def build_train_seqs_from_df(df_zone, train_years, seq_len, horizon=1):
    """Build (X_raw, y) sequence arrays from a zone DataFrame. Used for PCHIP."""
    d = df_zone.copy().sort_values("Datetime_UTC")
    for c in RAW_FEATURES + [TARGET]:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d = d.dropna(subset=RAW_FEATURES + [TARGET])
    d["year"] = d["Datetime_UTC"].dt.year
    tr = d[d["year"].isin(train_years)]

    X = tr[RAW_FEATURES].values.astype(np.float32)
    y = tr[TARGET].values.astype(np.float32)
    Xs, ys = [], []
    for i in range(seq_len - 1, len(tr) - horizon):
        Xs.append(X[i - seq_len + 1 : i + 1])
        ys.append(y[i + horizon])
    return np.stack(Xs), np.array(ys, dtype=np.float32)


def train_from_arrays(
    X_train_raw, y_train_raw,  # (N, seq_len, n_feat) original scale
    df_test_zone,              # full zone DataFrame — year==2025 used as test
    zone_name,
    seq_len=72, horizon=1,
    d_model=64, nhead=4, n_layers=2,
    batch_size=256, lr=1e-3, max_epochs=30, patience=5,
    label="",
):
    """Train the Transformer on pre-built arrays; evaluate on real 2025 data."""
    N, T, F = X_train_raw.shape

    # Scale features (fit on training set)
    feat_sc = _SS()
    X_tr_sc = feat_sc.fit_transform(
        X_train_raw.reshape(-1, F)
    ).reshape(N, T, F).astype(np.float32)

    # Build test sequences from real 2025 data
    dz = df_test_zone.copy().sort_values("Datetime_UTC")
    for c in RAW_FEATURES + [TARGET]:
        dz[c] = pd.to_numeric(dz[c], errors="coerce")
    dz = dz.dropna(subset=RAW_FEATURES + [TARGET])
    dz["year"] = dz["Datetime_UTC"].dt.year
    test = dz[dz["year"].eq(2025)].reset_index(drop=True)

    X_te_sc = feat_sc.transform(test[RAW_FEATURES].astype(float).values)
    y_te    = test[TARGET].astype(float).values
    t_te    = test["Datetime_UTC"].values
    ws_te   = test["is_wildfire_season"].astype(int).values

    Xte_l, yte_l, tte_l, wte_l = [], [], [], []
    for i in range(seq_len - 1, len(test) - horizon):
        Xte_l.append(X_te_sc[i - seq_len + 1 : i + 1])
        yte_l.append(y_te[i + horizon])
        tte_l.append(t_te[i + horizon])
        wte_l.append(ws_te[i + horizon])

    Xte = np.stack(Xte_l).astype(np.float32)
    yte = np.array(yte_l, dtype=np.float32)
    tte = np.array(tte_l)
    wte = np.array(wte_l, dtype=int)

    # Train / val split (last 10%, time-ordered)
    n_val = max(1, int(0.1 * N))
    X_tr, y_tr = X_tr_sc[:N - n_val], y_train_raw[:N - n_val]
    X_va, y_va = X_tr_sc[N - n_val:], y_train_raw[N - n_val:]

    tr_ld = DataLoader(SeqDataset(X_tr, y_tr, None),
                       batch_size=batch_size, shuffle=True,
                       pin_memory=(device.type == "cuda"))
    va_ld = DataLoader(SeqDataset(X_va, y_va, None),
                       batch_size=batch_size, shuffle=False,
                       pin_memory=(device.type == "cuda"))
    te_ld = DataLoader(SeqDataset(Xte, yte, tte, wildfire_flag=wte),
                       batch_size=batch_size, shuffle=False,
                       pin_memory=(device.type == "cuda"))

    model     = TransformerRegressor(n_features=F, d_model=d_model,
                                     nhead=nhead, num_layers=n_layers).to(device)
    opt       = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn   = nn.MSELoss()
    use_amp   = device.type == "cuda"
    amp_sc    = torch.amp.GradScaler("cuda", enabled=use_amp)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)

    best_val, best_state, no_improve = float("inf"), None, 0

    for ep in range(1, max_epochs + 1):
        model.train()
        for xb, yb, _ in tr_ld:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=use_amp):
                loss = loss_fn(model(xb), yb)
            amp_sc.scale(loss).backward()
            amp_sc.step(opt); amp_sc.update()

        model.eval()
        vl = 0.0
        with torch.no_grad():
            for xb, yb, _ in va_ld:
                xb, yb = xb.to(device), yb.to(device)
                with torch.amp.autocast("cuda", enabled=use_amp):
                    vl += loss_fn(model(xb), yb).item() * len(yb)
        vl /= max(len(y_va), 1)
        scheduler.step(vl)
        print(f"  [{label}|{zone_name}] ep {ep:02d} | val {vl:.4f} | "
              f"lr {opt.param_groups[0]['lr']:.1e}")

        if vl < best_val - 1e-6:
            best_val, no_improve = vl, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  Early stopping at ep {ep}")
                break

    model.load_state_dict(best_state)
    model.eval()

    yhat_c, ytrue_c, idx_c = [], [], []
    with torch.no_grad():
        for xb, yb, ix in te_ld:
            yhat_c.append(model(xb.to(device)).cpu().numpy())
            ytrue_c.append(yb.numpy())
            idx_c.append(ix.numpy())

    yhat  = np.concatenate(yhat_c)
    ytrue = np.concatenate(ytrue_c)
    idxs  = np.concatenate(idx_c)

    pred_df = pd.DataFrame({
        "Datetime_UTC":       tte[idxs],
        "y_true":             ytrue,
        "y_pred":             yhat,
        "is_wildfire_season": wte[idxs],
    }).sort_values("Datetime_UTC").reset_index(drop=True)

    met_all = {
        "Zone": zone_name, "Dataset": label,
        "N_train": N, "N_test": len(ytrue),
        **compute_metrics(ytrue, yhat, THRESHOLD),
    }
    ws = pred_df[pred_df["is_wildfire_season"].eq(1)]
    met_ws = {"Zone": zone_name, "Dataset": label}
    if len(ws) > 0:
        ws_m = compute_metrics(ws["y_true"].values, ws["y_pred"].values, THRESHOLD)
        met_ws.update({f"{k}_WS": v for k, v in ws_m.items()})

    return met_all, met_ws, pred_df


print("Helper functions ready.")


In [ ]:

# ── PCHIP 30-min Experiment ───────────────────────────────────────────────────
PCHIP_SEQ = 144   # 72 hours at 30-min resolution

pchip_csv = Path("..") / "Dataset" / "Outputs" / "CS2_pchip30min.csv"
df_pchip  = pd.read_csv(pchip_csv)
df_pchip["Datetime_UTC"] = pd.to_datetime(df_pchip["Datetime_UTC"], utc=True, errors="coerce")
df_pchip  = df_pchip.dropna(subset=["Datetime_UTC"])
for c in RAW_FEATURES:
    df_pchip[c] = pd.to_numeric(df_pchip[c], errors="coerce")
df_pchip = df_pchip.sort_values(["Zone", "Datetime_UTC"]).reset_index(drop=True)

print(f"PCHIP rows: {len(df_pchip):,}  |  zones: {sorted(df_pchip['Zone'].unique())}")

pchip_results_all = []
pchip_results_ws  = []
pchip_preds       = {}

for zone in zones:
    print(f"\n{'='*55}\nZone: {zone}")
    dz_p = df_pchip[df_pchip["Zone"] == zone].copy()

    # Build training sequences at 30-min resolution (SEQ_LEN=144)
    X_train, y_train = build_train_seqs_from_df(
        dz_p, train_years=[2022, 2023, 2024],
        seq_len=PCHIP_SEQ, horizon=1,
    )
    print(f"  Training seqs: {len(X_train):,}  shape: {X_train.shape}")

    # Test sequences also built from PCHIP 2025 data (30-min resolution)
    met_all, met_ws, pred_df = train_from_arrays(
        X_train, y_train,
        df_test_zone=dz_p,      # PCHIP df covers 2025 at 30-min too
        zone_name=zone,
        seq_len=PCHIP_SEQ, horizon=1,
        label="PCHIP",
    )
    pchip_results_all.append(met_all)
    pchip_results_ws.append(met_ws)
    pchip_preds[zone] = pred_df

pd.DataFrame(pchip_results_all)


In [ ]:

# ── CVAE Augmented Experiment ─────────────────────────────────────────────────
cvae_path = Path("..") / "Dataset" / "Outputs" / "CS2_cvae_aug.npz"
cvae_data  = np.load(cvae_path)

print(f"CVAE NPZ keys: {list(cvae_data.keys())}")

cvae_results_all = []
cvae_results_ws  = []
cvae_preds       = {}

for zone in zones:
    key = zone.replace(" ", "_")
    print(f"\n{'='*55}\nZone: {zone}")

    X_real = cvae_data[f"{key}_X_real"]
    y_real = cvae_data[f"{key}_y_real"]
    X_syn  = cvae_data[f"{key}_X_syn"]
    y_syn  = cvae_data[f"{key}_y_syn"]

    X_aug = np.concatenate([X_real, X_syn])
    y_aug = np.concatenate([y_real, y_syn])
    print(f"  Real: {len(X_real):,}  Synthetic: {len(X_syn):,}  Total: {len(X_aug):,}")

    # Shuffle so synthetic seqs are interspersed with real during training
    rng   = np.random.default_rng(42)
    perm  = rng.permutation(len(X_aug))
    X_aug, y_aug = X_aug[perm], y_aug[perm]

    dz_orig = df[df["Zone"] == zone].copy()

    met_all, met_ws, pred_df = train_from_arrays(
        X_aug, y_aug,
        df_test_zone=dz_orig,   # test on real hourly 2025 data
        zone_name=zone,
        seq_len=72, horizon=1,
        label="CVAE",
    )
    cvae_results_all.append(met_all)
    cvae_results_ws.append(met_ws)
    cvae_preds[zone] = pred_df

pd.DataFrame(cvae_results_all)


In [ ]:

# ── TimeGAN Augmented Experiment ─────────────────────────────────────────────
tgan_path = Path("..") / "Dataset" / "Outputs" / "CS2_timegan_aug.npz"
tgan_data  = np.load(tgan_path)

print(f"TimeGAN NPZ keys: {list(tgan_data.keys())}")

timegan_results_all = []
timegan_results_ws  = []
timegan_preds       = {}

for zone in zones:
    key = zone.replace(" ", "_")
    print(f"\n{'='*55}\nZone: {zone}")

    X_real = tgan_data[f"{key}_X_real"]
    y_real = tgan_data[f"{key}_y_real"]
    X_syn  = tgan_data[f"{key}_X_syn"]
    y_syn  = tgan_data[f"{key}_y_syn"]

    X_aug = np.concatenate([X_real, X_syn])
    y_aug = np.concatenate([y_real, y_syn])
    print(f"  Real: {len(X_real):,}  Synthetic: {len(X_syn):,}  Total: {len(X_aug):,}")

    rng   = np.random.default_rng(42)
    perm  = rng.permutation(len(X_aug))
    X_aug, y_aug = X_aug[perm], y_aug[perm]

    dz_orig = df[df["Zone"] == zone].copy()

    met_all, met_ws, pred_df = train_from_arrays(
        X_aug, y_aug,
        df_test_zone=dz_orig,
        zone_name=zone,
        seq_len=72, horizon=1,
        label="TimeGAN",
    )
    timegan_results_all.append(met_all)
    timegan_results_ws.append(met_ws)
    timegan_preds[zone] = pred_df

pd.DataFrame(timegan_results_all)


In [ ]:

# ── Comparison table: all methods ────────────────────────────────────────────
keep_cols = ["Zone", "Dataset", "N_train", "N_test",
             "RMSE", "MAE", "R2", "Precision@25", "Recall@25", "F1@25",
             "ActualEvents", "PredEvents"]

# Hardcoded original best results (from prior grid search outputs saved in notebook)
# LFV best: SEQ=72, H=1 | SI best: SEQ=168, H=1
_orig_hardcoded = {
    "Lower Fraser Valley": {"RMSE": 0.6041, "MAE": 0.3309, "R2": 0.9534,
                            "Precision@25": 0.0, "Recall@25": 0.0, "F1@25": 0.0,
                            "ActualEvents": 13, "PredEvents": 0},
    "Southern Interior":   {"RMSE": 1.1925, "MAE": 0.7864, "R2": 0.9406,
                            "Precision@25": 0.0, "Recall@25": 0.0, "F1@25": 0.0,
                            "ActualEvents": 70, "PredEvents": 0},
}

orig_rows = []
for zone in zones:
    if "all_df" in dir() and len(all_df) > 0:
        # Use freshly trained results if available
        best = (all_df[(all_df["Zone"] == zone) & (all_df["HORIZON"] == 1)]
                .sort_values("RMSE").iloc[0])
        n_test = len(preds_best[zone]["pred_df"]) if "preds_best" in dir() else 0
        orig_rows.append({
            "Zone": zone, "Dataset": "Original (H=1)",
            "N_train": 26175, "N_test": n_test,
            "RMSE":         round(float(best["RMSE"]), 4),
            "MAE":          round(float(best["MAE"]),  4),
            "R2":           round(float(best["R2"]),   4),
            "Precision@25": round(float(best["Precision@25"]), 4),
            "Recall@25":    round(float(best["Recall@25"]),    4),
            "F1@25":        round(float(best["F1@25"]),        4),
            "ActualEvents": int(best["ActualEvents"]),
            "PredEvents":   int(best["PredEvents"]),
        })
    else:
        # Fallback to hardcoded values from prior run
        h = _orig_hardcoded[zone]
        orig_rows.append({"Zone": zone, "Dataset": "Original (H=1)",
                          "N_train": 26175, "N_test": 0, **h})

frames = [
    pd.DataFrame(orig_rows),
    pd.DataFrame(pchip_results_all),
    pd.DataFrame(cvae_results_all),
    pd.DataFrame(timegan_results_all),
]

cmp_df = (pd.concat(frames, ignore_index=True)
            .sort_values(["Zone", "Dataset"])
            .reset_index(drop=True))

for col in ["RMSE", "MAE", "R2", "Precision@25", "Recall@25", "F1@25"]:
    cmp_df[col] = pd.to_numeric(cmp_df[col], errors="coerce").round(4)

print("Full 2025 Test Set — Augmentation Method Comparison")
print("(PCHIP N_test is ~2x others due to 30-min resolution)")
display(cmp_df[keep_cols])

# Wildfire-season-only table
keep_ws = ["Zone", "Dataset", "RMSE_WS", "MAE_WS", "R2_WS",
           "Precision@25_WS", "Recall@25_WS", "F1@25_WS",
           "ActualEvents_WS", "PredEvents_WS"]

ws_frames = [
    pd.DataFrame(pchip_results_ws),
    pd.DataFrame(cvae_results_ws),
    pd.DataFrame(timegan_results_ws),
]
ws_df2 = (pd.concat(ws_frames, ignore_index=True)
            .sort_values(["Zone", "Dataset"])
            .reset_index(drop=True))
for col in [c for c in keep_ws if c not in ("Zone", "Dataset")]:
    if col in ws_df2.columns:
        ws_df2[col] = pd.to_numeric(ws_df2[col], errors="coerce").round(4)

print("\nWildfire Season Only")
display(ws_df2[[c for c in keep_ws if c in ws_df2.columns]])


In [7]:

# ── Side-by-side visual comparison ───────────────────────────────────────────
MAX_H = 24 * 21   # first 21 days of 2025 per panel

# Unify all pred dicts to {zone: pred_df}
all_preds = {
    "Original\n(hourly, H=1)": {z: preds_best[z]["pred_df"] for z in zones},
    "PCHIP 30min\n(144-step seq)": pchip_preds,
    "CVAE Aug\n(1× synthetic)":    cvae_preds,
    "TimeGAN Aug\n(1× synthetic)": timegan_preds,
}

n_rows = len(zones)
n_cols = len(all_preds)
fig, axes = plt.subplots(n_rows, n_cols,
                          figsize=(7 * n_cols, 4 * n_rows),
                          sharey="row")

for r, zone in enumerate(zones):
    for c, (label, preds_dict) in enumerate(all_preds.items()):
        ax = axes[r, c]
        d  = preds_dict[zone].sort_values("Datetime_UTC").copy()

        # Resample PCHIP to hourly so all panels show the same x-density
        if "PCHIP" in label:
            d = (d.set_index("Datetime_UTC")
                  .resample("1h").mean()
                  .reset_index())

        if MAX_H and len(d) > MAX_H:
            d = d.iloc[:MAX_H]

        ax.plot(d["Datetime_UTC"], d["y_true"],
                label="Actual",    linewidth=1.0, alpha=0.9,  color="steelblue")
        ax.plot(d["Datetime_UTC"], d["y_pred"],
                label="Predicted", linewidth=1.0, alpha=0.85, color="darkorange")
        ax.axhline(THRESHOLD, linestyle="--", linewidth=0.8,
                   color="red", alpha=0.7, label=f"Alert={THRESHOLD}")

        ax.set_title(f"{zone}\n{label}", fontsize=9)
        ax.set_ylabel("PM2.5 (µg/m³)" if c == 0 else "")
        ax.grid(alpha=0.25)
        ax.tick_params(axis="x", rotation=25, labelsize=7)
        if r == 0 and c == 0:
            ax.legend(fontsize=7, loc="upper right")

fig.suptitle(
    "Actual vs Predicted — Augmentation Method Comparison  |  2025 Test, first 21 days",
    fontsize=12, y=1.01,
)
plt.tight_layout()
plt.show()


NameError: name 'preds_best' is not defined